In [1]:
import os
import csv
import numpy as np

from concurrent.futures import ThreadPoolExecutor, as_completed
from skimage.io import imread
from skimage import img_as_float, transform
from skimage.segmentation import slic
from skimage.color import rgb2hsv

from scipy.stats import circvar
from sklearn.cluster import KMeans


def load_image(path: str, max_size: int = 512) -> np.ndarray:
    img = img_as_float(imread(path))
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if img.shape[-1] == 4:
        img = img[:, :, :3]

    if max_size is not None:
        h, w = img.shape[:2]
        scale = max_size / max(h, w)
        if scale < 1.0:
            img = transform.resize(
                img,
                (int(h * scale), int(w * scale)),
                anti_aliasing=True,
            )

    return img


def get_segment_features(
    image: np.ndarray,
    hsv: np.ndarray,
    segments: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    seg_ids = np.unique(segments)
    n = len(seg_ids)

    lut = np.zeros(segments.max() + 1, dtype=np.intp)
    lut[seg_ids] = np.arange(n)
    flat_idx = lut[segments.ravel()]

    counts = np.bincount(flat_idx, minlength=n).astype(np.float64)

    rgb_flat = image.reshape(-1, 3)
    rgb_sum = np.zeros((n, 3))
    np.add.at(rgb_sum, flat_idx, rgb_flat)
    rgb_means = rgb_sum / counts[:, None]

    hsv_flat = hsv.reshape(-1, 3)

    sv_sum = np.zeros((n, 2))
    np.add.at(sv_sum, flat_idx, hsv_flat[:, 1:])
    sv_means = sv_sum / counts[:, None]

    angles = hsv_flat[:, 0] * 2 * np.pi
    sin_sum = np.zeros(n)
    cos_sum = np.zeros(n)
    np.add.at(sin_sum, flat_idx, np.sin(angles))
    np.add.at(cos_sum, flat_idx, np.cos(angles))
    h_means = (np.arctan2(sin_sum / counts, cos_sum / counts) % (2 * np.pi)) / (2 * np.pi)

    hsv_means = np.column_stack([h_means, sv_means])
    return rgb_means, hsv_means


def calculate_variances(means: np.ndarray) -> tuple:
    return np.var(means, axis=0) if len(means) >= 2 else (0.0, 0.0, 0.0)


def calculate_hsv_variances(means: np.ndarray) -> tuple:
    if len(means) < 2:
        return (0.0, 0.0, 0.0)
    return (circvar(means[:, 0], high=1, low=0), np.var(means[:, 1]), np.var(means[:, 2]))


def color_dominance(
    hsv_flat: np.ndarray,
    clusters: int = 5,
    pixel_step: int = 4,
) -> list[tuple[float, np.ndarray]]:
    sample = hsv_flat[::pixel_step]
    kmeans = KMeans(n_clusters=clusters, n_init=10, random_state=0)
    kmeans.fit(sample)

    _, counts = np.unique(kmeans.labels_, return_counts=True)
    ratios = counts / len(kmeans.labels_)

    return sorted(zip(ratios, kmeans.cluster_centers_), key=lambda x: x[0], reverse=True)


def process_image(args: tuple[str, str, int, int, int]) -> dict | None:
    filename, path, max_size, n_segments, pixel_step = args
    try:
        image    = load_image(path, max_size=max_size)
        hsv      = rgb2hsv(image)
        hsv_flat = hsv.reshape(-1, 3)
        segments = slic(image, n_segments=n_segments, compactness=10, start_label=1)
        rgb_m, hsv_m = get_segment_features(image, hsv, segments)

        return {
            "file":            filename,
            "rgb_means":       rgb_m,
            "hsv_means":       hsv_m,
            "rgb_var":         calculate_variances(rgb_m),
            "hsv_var":         calculate_hsv_variances(hsv_m),
            "dominant_colors": color_dominance(hsv_flat, pixel_step=pixel_step),
        }
    except Exception as e:
        print(f"[WARN] Skipping {filename}: {e}")
        return None


def save_csv(results: list[dict], output_path: str) -> None:
    with open(output_path, "w", newline="") as f:
        writer = csv.writer(f)

        writer.writerow([
            "file",
            "rgb_var_r", "rgb_var_g", "rgb_var_b",
            "hsv_var_h", "hsv_var_s", "hsv_var_v",
            *[f"dom_color_{i+1}_ratio" for i in range(5)],
            *[f"dom_color_{i+1}_h" for i in range(5)],
            *[f"dom_color_{i+1}_s" for i in range(5)],
            *[f"dom_color_{i+1}_v" for i in range(5)],
        ])

        for r in results:
            rgb_v = r["rgb_var"]
            hsv_v = r["hsv_var"]
            dom   = r["dominant_colors"]

            ratios = [d[0] for d in dom]
            colors = [d[1] for d in dom]

            writer.writerow([
                r["file"],
                *rgb_v,
                *hsv_v,
                *ratios,
                *[c[0] for c in colors],
                *[c[1] for c in colors],
                *[c[2] for c in colors],
            ])


VALID_EXT = {".jpg", ".jpeg", ".png"}

def process_folder(
    folder: str,
    output_csv: str = "results.csv",
    max_size: int = 512,
    n_segments: int = 50,
    pixel_step: int = 4,
    workers: int = None,
) -> list[dict]:
    paths = [
        (fn, os.path.join(folder, fn))
        for fn in os.listdir(folder)
        if os.path.splitext(fn)[1].lower() in VALID_EXT
    ]

    args = [(fn, p, max_size, n_segments, pixel_step) for fn, p in paths]

    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(process_image, a): a[0] for a in args}
        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                results.append(result)

    save_csv(results, output_csv)
    print(f"Saved {len(results)} rows to {output_csv}")

    return results


results = process_folder(
    "images/",
    output_csv="results.csv",
    max_size=512,
    n_segments=50,
    pixel_step=4,
)

c:\Users\Wardz\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Wardz\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\Wardz\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Wardz\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Saved 2298 rows to results.csv
